# av-ecs-poc – build and push the container image

Builds the POC image in two layers, the way you would deploy the vendor's images:

1. **app image** (`../app`): plays the role of the *unmodified vendor image* (roles director, server, guard, agent). It only reads `NG_SERVER_NAME` / `GUARD_EXTERNAL_IP`.
2. **wrapper image** (`../wrapper`): `FROM` the app image, adds `entrypoint.sh`, which sets those variables to the task IP and then starts the original command. This is the image that is pushed and deployed.

Then it runs a quick smoke test, and pushes it to the ECR repository created by the `av-ecs-poc-base` stack.

**Run this after** the base stack is deployed and **before** the services stack (or rerun it to ship a new version).

Requirements:
- Docker available to the notebook. A SageMaker **notebook instance** has it. **SageMaker Studio** needs Docker access enabled on the domain, plus `STUDIO = True` below.
- IAM permissions for the notebook role: `cloudformation:DescribeStacks`, `ecr:GetAuthorizationToken`, push actions on the repository, `kms:GenerateDataKey`/`kms:Decrypt` on the project key (the repository is KMS-encrypted), and optionally `ecs:UpdateService`.

## 1. Settings

In [ ]:
import os

REGION      = "us-east-1"
PROJECT     = "av-ecs-poc"
BASE_STACK  = f"{PROJECT}-base"
SVC_STACK   = f"{PROJECT}-services"
TAG         = "v1"                  # bump (v2, v3...) for each new build, then update ImageTag on the services stack

# Base image and pip index - point these at internal mirrors if public sources are blocked
BASE_IMAGE    = "public.ecr.aws/docker/library/python:3.12-slim"
PIP_INDEX_URL = ""                  # e.g. "https://artifactory.example.internal/api/pypi/pypi/simple" (blank = default PyPI)

STUDIO   = False                    # True when running in SageMaker Studio (adds --network sagemaker)
APP_DIR     = os.path.abspath("../app")       # "vendor" image sources
WRAPPER_DIR = os.path.abspath("../wrapper")   # address wrapper layer

for d in (APP_DIR, WRAPPER_DIR):
    assert os.path.isfile(os.path.join(d, "Dockerfile")), f"Dockerfile not found in {d}"
print("App dir:    ", APP_DIR)
print("Wrapper dir:", WRAPPER_DIR)

## 2. Helpers

In [ ]:
import shlex, subprocess, sys

def run(cmd, check=True, input_text=None, quiet=False):
    """Run a shell command and stream its output into the notebook."""
    if not quiet:
        print("$", cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd))
    proc = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            stdin=subprocess.PIPE if input_text else None, text=True)
    if input_text:
        proc.stdin.write(input_text); proc.stdin.close()
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet:
            sys.stdout.write(line)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")
    return "".join(lines)

NET = ["--network", "sagemaker"] if STUDIO else []

## 3. Check Docker

In [ ]:
run("docker version --format 'client {{.Client.Version}} / server {{.Server.Version}}'")

## 4. Look up the ECR repository from the base stack

In [ ]:
import boto3

session = boto3.Session(region_name=REGION)
cfn, ecr, ecs = session.client("cloudformation"), session.client("ecr"), session.client("ecs")

outputs = {o["OutputKey"]: o["OutputValue"]
           for o in cfn.describe_stacks(StackName=BASE_STACK)["Stacks"][0]["Outputs"]}
REPO_URI  = outputs["RepositoryUri"]
REGISTRY  = REPO_URI.split("/")[0]
REPO_NAME = REPO_URI.split("/", 1)[1]
IMAGE     = f"{REPO_URI}:{TAG}"
print("Repository:", REPO_URI)
print("Image     :", IMAGE)

## 5. Log in to ECR

In [ ]:
import base64

auth = ecr.get_authorization_token()["authorizationData"][0]
user, password = base64.b64decode(auth["authorizationToken"]).decode().split(":", 1)
run(["docker", "login", "--username", user, "--password-stdin", REGISTRY], input_text=password)

## 6. Build

Both layers are built for `linux/amd64` to match the Fargate task definitions (`X86_64`). Only the wrapper image is pushed.

In [ ]:
# Make sure we build the right sources, with Unix line endings
import glob

for d in (APP_DIR, WRAPPER_DIR):
    print("Sources:", d)
    for f in sorted(glob.glob(os.path.join(d, "*"))):
        if os.path.isfile(f):
            raw = open(f, "rb").read()
            fixed = raw.removeprefix(b"\xef\xbb\xbf").replace(b"\r\n", b"\n")
            if fixed != raw:
                open(f, "wb").write(fixed)
                print(f"  normalised line endings: {os.path.basename(f)}")
            else:
                print(f"  ok: {os.path.basename(f)}")
assert os.path.isfile(os.path.join(WRAPPER_DIR, "entrypoint.sh")), "wrapper/entrypoint.sh missing - use the latest package"
print("Sources look current")

In [ ]:
VENDOR_LOCAL = f"{PROJECT}-vendor:{TAG}"   # local only, never pushed

# 1) the "vendor" image - unchanged application
run(["docker", "build", *NET, "--platform", "linux/amd64",
     "--build-arg", f"BASE_IMAGE={BASE_IMAGE}", "--build-arg", f"PIP_INDEX_URL={PIP_INDEX_URL}",
     "-t", VENDOR_LOCAL, APP_DIR])
print("Original start command:",
      run(["docker", "image", "inspect", VENDOR_LOCAL, "--format", "{{json .Config.Entrypoint}} {{json .Config.Cmd}} user={{.Config.User}}"], quiet=True).strip())

# 2) the wrapper layer on top - this is what ECS runs
run(["docker", "build", *NET, "--platform", "linux/amd64",
     "--build-arg", f"VENDOR_IMAGE={VENDOR_LOCAL}", "--build-arg", "VENDOR_USER=10001",
     "-t", IMAGE, WRAPPER_DIR])

### Check the image architecture
Must be `linux/amd64` for the default task definitions (`CpuArchitecture=X86_64`). A mismatch makes every task stop with `exec format error`.

In [ ]:
arch = run(["docker", "image", "inspect", IMAGE, "--format", "{{.Os}}/{{.Architecture}}"], quiet=True).strip()
print("Image platform:", arch)
assert arch == "linux/amd64", (
    f"Built {arch}. Enable buildx/QEMU and rebuild for linux/amd64, "
    "or deploy the services stack with CpuArchitecture=ARM64.")

## 7. Smoke test

Starts the wrapped image as a **guard** (needs no registry) and checks that the wrapper ran (`address_resolved` line), then started the original process (`guard_start`), and that the tunnel port answers. Outside ECS the wrapper may find no task IP; that's expected here.

In [ ]:
import time

name = f"{PROJECT}-smoke"
print("Entrypoint baked into the image:",
      run(["docker", "image", "inspect", IMAGE, "--format", "{{json .Config.Entrypoint}}"], quiet=True).strip())
run(["docker", "rm", "-f", name], check=False, quiet=True)
run(["docker", "run", "-d", *NET, "--name", name, "-e", "ROLE=guard", "-e", "GUARD_API_ROOT=http://127.0.0.1:1", IMAGE])
try:
    time.sleep(5)
    state = run(["docker", "inspect", name, "--format", "{{.State.Status}} exit={{.State.ExitCode}}"], quiet=True).strip()
    logs = run(["docker", "logs", name], check=False, quiet=True)
    print("Container state:", state)
    print("---- container logs ----")
    print(logs or "(no output)")
    assert state.startswith("running"), "container exited - see the logs above"
    entry = run(["docker", "image", "inspect", IMAGE, "--format", "{{json .Config.Entrypoint}}"], quiet=True).strip()
    assert entry == '["/bin/sh","/wrapper/entrypoint.sh"]', f"unexpected entrypoint {entry}"
    run(["docker", "exec", name, "python", "/app/healthcheck.py"])   # exits non-zero if port 9000 is closed
    assert "address_resolved" in logs, "wrapper entrypoint did not run"
    assert "guard_start" in logs, "the original (vendor) process did not start after the wrapper"
    print("Smoke test passed")
finally:
    run(["docker", "rm", "-f", name], check=False, quiet=True)

## 8. Push

In [ ]:
run(["docker", "push", IMAGE])

## 9. Verify in ECR

The repository scans on push. The findings summary may take a minute to appear.

In [ ]:
img = ecr.describe_images(repositoryName=REPO_NAME, imageIds=[{"imageTag": TAG}])["imageDetails"][0]
print("Digest :", img["imageDigest"])
print("Size MB:", round(img["imageSizeInBytes"] / 1024 / 1024, 1))
print("Pushed :", img["imagePushedAt"])
print("Scan   :", img.get("imageScanStatus", {}).get("status"),
      img.get("imageScanFindingsSummary", {}).get("findingSeverityCounts", {}))

## 10. Roll out (optional)

- **First time:** deploy `cfn/02-services.yaml` with `ImageTag` set to the `TAG` above (`scripts/deploy.sh` or the console).
- **New tag** (v2, v3...): update the services stack parameter `ImageTag`.
- **Same tag rebuilt:** run the cell below to force the ECS services to pull the image again.

Forcing a new deployment of **server** replaces every server task: each comes back with a new IP, and the devices are rehomed (README scenario 5).

In [ ]:
FORCE_REDEPLOY = False          # set True to roll the running services onto the rebuilt image
SERVICES = ["server", "director", "guard", "agent"]

if FORCE_REDEPLOY:
    for s in SERVICES:
        svc = f"{PROJECT}-{s}"
        ecs.update_service(cluster=PROJECT, service=svc, forceNewDeployment=True)
        print("Redeploying", svc)
    ecs.get_waiter("services_stable").wait(cluster=PROJECT, services=[f"{PROJECT}-{s}" for s in SERVICES])
    print("All services stable")
else:
    print("Skipped - set FORCE_REDEPLOY = True to redeploy")